In [0]:
# ============================================================
# NOTEBOOK 02 — TRUSTED: FEATURE ENGINEERING
# Vermont School - Early Warning System V2
#
# INPUT:  bronze/prepared/24_25/ and bronze/prepared/25_26/
# OUTPUT: trusted/train_dataset/   ← 24-25 (T1+T2 features, risk_level target)
#         trusted/predict_dataset/ ← 25-26 (T1+T2 features only)
# ============================================================

BRONZE  = "/Volumes/workspace/vermont/bronze"
TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

PREP_24 = f"{BRONZE}/prepared/24_25"
PREP_25 = f"{BRONZE}/prepared/25_26"

TRAIN_PATH   = f"{TRUSTED}/train_dataset"
PREDICT_PATH = f"{TRUSTED}/predict_dataset"

for path in [TRAIN_PATH, PREDICT_PATH]:
    dbutils.fs.mkdirs(path)

# Subject groups
GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

# Features used for training — only T1 and T2
# (what we know MID-YEAR, before final grades)
FEATURES_T1_T2 = (
    [f'{g}_T1' for g in GROUPS] +
    [f'{g}_T2' for g in GROUPS] +
    ['total_absences', 'absence_class', 'absence_school',
     'late', 'early_leave', 'n_f1', 'n_f2', 'n_fault_types']
)

TARGET = 'risk_level'

print("✓ Configuration loaded")
print(f"  Features (T1+T2 only): {len(FEATURES_T1_T2)}")
print(f"  Target: {TARGET}")
print(f"\n  Key design decision:")
print(f"  → Model trained with T1+T2 only (mid-year snapshot)")
print(f"  → Target is FINAL risk level (end of year)")
print(f"  → This enables genuine early warning predictions")

In [0]:
# CELDA 2 — Build training and prediction datasets

import pandas as pd
import numpy as np

# Load both years
df_24 = spark.read.parquet(PREP_24).toPandas()
df_25 = spark.read.parquet(PREP_25).toPandas()

print(f"Loaded 24-25: {len(df_24)} students")
print(f"Loaded 25-26: {len(df_25)} students")

# ── TRAINING DATASET (24-25) ──
# Features: T1 + T2 grades + attendance + disciplinary
# Target: final risk level (calculated from cumulative grades)

# Check available features
missing_train = [f for f in FEATURES_T1_T2 if f not in df_24.columns]
if missing_train:
    print(f"\n  Warning — missing features in 24-25: {missing_train}")

available_features = [f for f in FEATURES_T1_T2 if f in df_24.columns]

# Build training set
df_train = df_24[['student_id', 'grade', 'section_anon', 'year'] + 
                  available_features + [TARGET]].copy()

# Fill NaN with 0 for disciplinary, keep NaN for grades
disc_cols = ['total_absences','absence_class','absence_school',
             'late','early_leave','n_f1','n_f2','n_fault_types']
df_train[disc_cols] = df_train[disc_cols].fillna(0)

print(f"\n── TRAINING DATASET (24-25) ──")
print(f"  Students: {len(df_train)}")
print(f"  Features: {len(available_features)}")
print(f"  Target distribution:")
print(df_train[TARGET].value_counts().to_string())

# ── PREDICTION DATASET (25-26) ──
# Same features, T1 + T2 only
# Also include T3 partial for alert confirmation later

t3_cols = [f'{g}_T3' for g in GROUPS if f'{g}_T3' in df_25.columns]

df_predict = df_25[['student_id', 'grade', 'section_anon', 'year'] +
                    available_features + t3_cols +
                    ['risk_level', 'avg_cumulative', 
                     'min_cumulative', 'n_subjects_below_4']].copy()

df_predict[disc_cols] = df_predict[disc_cols].fillna(0)

print(f"\n── PREDICTION DATASET (25-26) ──")
print(f"  Students: {len(df_predict)}")
print(f"  Features: {len(available_features)}")
print(f"  T3 partial columns included: {len(t3_cols)}")
print(f"  (T3 used only for alert confirmation, NOT for prediction)")

# ── SAVE TO TRUSTED ──
spark.createDataFrame(df_train).write.mode("overwrite").parquet(TRAIN_PATH)
spark.createDataFrame(df_predict).write.mode("overwrite").parquet(PREDICT_PATH)

print(f"\n✓ Saved to Trusted:")
print(f"  {TRAIN_PATH}")
print(f"  {PREDICT_PATH}")

In [0]:
# CELDA 3 — Validation and summary

df_train_check   = spark.read.parquet(TRAIN_PATH).toPandas()
df_predict_check = spark.read.parquet(PREDICT_PATH).toPandas()

print("=" * 55)
print("TRUSTED DATASETS — VALIDATION SUMMARY")
print("=" * 55)

# Check nulls in key features
print("\n── Training dataset nulls (key features) ──")
key_features = [f'{g}_T1' for g in GROUPS[:5]] + \
               [f'{g}_T2' for g in GROUPS[:5]] + \
               ['total_absences', 'n_f1', 'n_f2']
nulls = df_train_check[key_features].isnull().sum()
print(nulls[nulls > 0].to_string() if nulls[nulls > 0].any() 
      else "  No nulls in key features ✓")

print("\n── Prediction dataset nulls (key features) ──")
nulls2 = df_predict_check[key_features].isnull().sum()
print(nulls2[nulls2 > 0].to_string() if nulls2[nulls2 > 0].any() 
      else "  No nulls in key features ✓")

# Grade distribution
print("\n── Students per grade ──")
print("  Training (24-25):")
print(df_train_check['grade'].value_counts().sort_index().to_string())
print("  Prediction (25-26):")
print(df_predict_check['grade'].value_counts().sort_index().to_string())

# Feature preview
print(f"\n── Feature ranges (training) ──")
cols_check = ['Science_T1','Science_T2','Mathematics_T1',
              'Mathematics_T2','total_absences','n_f1']
print(df_train_check[cols_check].describe().round(2).to_string())

print(f"\n✓ Notebook 02 complete — Trusted datasets ready")
print(f"  Training:   {TRAIN_PATH}")
print(f"  Prediction: {PREDICT_PATH}")

In [0]:
# Revisar columnas T3 disponibles
t3_disponibles = [col for col in df_predict_check.columns if '_T3' in col]
print("Columnas T3 disponibles:")
print(t3_disponibles)
print(f"\nTotal: {len(t3_disponibles)}")

# Ver también todas las columnas del dataset de predicción
print("\nTodas las columnas del predict dataset:")
print(list(df_predict_check.columns))

In [0]:
# CELDA 4 — Feature Engineering: nuevas variables para clasificación, regresión y clustering
# Recargar explícitamente desde parquet
df_train_check   = spark.read.parquet(TRAIN_PATH).toPandas()
df_predict_check = spark.read.parquet(PREDICT_PATH).toPandas()

print(f"Train recargado: {len(df_train_check)} estudiantes")
print(f"Predict recargado: {len(df_predict_check)} estudiantes")

print("=" * 55)
print("FEATURE ENGINEERING — Variables derivadas")
print("=" * 55)

def add_engineered_features(df, groups, is_prediction=False):
    df = df.copy()
    
    # ── 1. Delta T1→T2 por asignatura ──
    # Captura si el estudiante está mejorando o deteriorándose
    for g in groups:
        t1_col = f'{g}_T1'
        t2_col = f'{g}_T2'
        if t1_col in df.columns and t2_col in df.columns:
            df[f'{g}_delta'] = df[t2_col] - df[t1_col]
    
    delta_cols = [f'{g}_delta' for g in groups 
                  if f'{g}_delta' in df.columns]
    print(f"  Deltas calculados: {len(delta_cols)}")

    # ── 2. Tendencia general T1→T2 ──
    # Promedio de todos los deltas — ¿sube o baja globalmente?
    if delta_cols:
        df['tendencia_general'] = df[delta_cols].mean(axis=1)
    
    # ── 3. Promedio general T1 y T2 ──
    t1_cols = [f'{g}_T1' for g in groups if f'{g}_T1' in df.columns]
    t2_cols = [f'{g}_T2' for g in groups if f'{g}_T2' in df.columns]
    df['avg_T1'] = df[t1_cols].mean(axis=1)
    df['avg_T2'] = df[t2_cols].mean(axis=1)

    # ── 4. Materias en bajo por trimestre ──
    # Cuántas materias < 4.0 en T1 y en T2
    df['n_bajo_T1'] = (df[t1_cols] < 4.0).sum(axis=1)
    df['n_bajo_T2'] = (df[t2_cols] < 4.0).sum(axis=1)
    
    # ── 5. Cambio en materias en bajo T1→T2 ──
    # Positivo = empeoró (más materias en bajo), Negativo = mejoró
    df['delta_materias_bajo'] = df['n_bajo_T2'] - df['n_bajo_T1']

    # ── 6. Materias destacadas (> 6.0) en T2 ──
    # Solo si no tiene ninguna en bajo — perfil sobresaliente
    df['n_destacadas_T2'] = (df[t2_cols] > 6.0).sum(axis=1)

    # ── 7. Dispersión de notas en T2 ──
    # Std alta = perfil irregular (bien en unas, mal en otras)
    # Std baja = perfil consistente (parejo en todo)
    df['dispersion_T2'] = df[t2_cols].std(axis=1)

    # ── 8. Ratio ausencias clase / total ──
    # Distingue si falta a clases específicas o al colegio en general
    if 'absence_class' in df.columns and 'total_absences' in df.columns:
        df['ratio_ausencia_clase'] = np.where(
            df['total_absences'] > 0,
            df['absence_class'] / df['total_absences'],
            0
        )

    # ── 9. Índice disciplinario ponderado ──
    # F2 pesa más que F1 por gravedad
    if 'n_f1' in df.columns and 'n_f2' in df.columns:
        df['indice_disciplinario'] = df['n_f1'] * 1 + df['n_f2'] * 3

    # ── 10. Nota mínima en T2 (materia más débil) ──
    df['min_nota_T2'] = df[t2_cols].min(axis=1)
    df['min_nota_T1'] = df[t1_cols].min(axis=1)

    # ── 11. Para predicción: nota mínima T3 necesaria por asignatura ──
    # fórmula: (4.0 - T1*0.30 - T2*0.30) / 0.40
    if is_prediction:
        for g in groups:
            t1 = f'{g}_T1'
            t2 = f'{g}_T2'
            if t1 in df.columns and t2 in df.columns:
                df[f'{g}_min_T3'] = (
                    (4.0 - df[t1] * 0.30 - df[t2] * 0.30) / 0.40
                ).clip(1.0, 7.0)
        print(f"  Notas mínimas T3 calculadas por asignatura ✓")

    return df

# Aplicar a ambos datasets
df_train_fe    = add_engineered_features(df_train_check, GROUPS, is_prediction=False)
df_predict_fe  = add_engineered_features(df_predict_check, GROUPS, is_prediction=True)

# ── Resumen de nuevas features ──
new_features = [
    'tendencia_general', 'avg_T1', 'avg_T2',
    'n_bajo_T1', 'n_bajo_T2', 'delta_materias_bajo',
    'n_destacadas_T2', 'dispersion_T2',
    'ratio_ausencia_clase', 'indice_disciplinario',
    'min_nota_T2', 'min_nota_T1'
]

print(f"\n── Nuevas features creadas ──")
print(f"  Deltas por asignatura:     {len(GROUPS)}")
print(f"  Variables derivadas:       {len(new_features)}")
print(f"  Total features nuevas:     {len(GROUPS) + len(new_features)}")

print(f"\n── Estadísticas de features clave (training) ──")
check_cols = ['tendencia_general', 'n_bajo_T1', 'n_bajo_T2', 
              'delta_materias_bajo', 'dispersion_T2', 'indice_disciplinario']
print(df_train_fe[check_cols].describe().round(2).to_string())

# ── Guardar versiones enriquecidas ──
TRAIN_FE_PATH   = f"{TRUSTED}/train_dataset_fe"
PREDICT_FE_PATH = f"{TRUSTED}/predict_dataset_fe"

spark.createDataFrame(df_train_fe).write.mode("overwrite").parquet(TRAIN_FE_PATH)
spark.createDataFrame(df_predict_fe).write.mode("overwrite").parquet(PREDICT_FE_PATH)

print(f"\n✓ Datasets enriquecidos guardados:")
print(f"  {TRAIN_FE_PATH}")
print(f"  {PREDICT_FE_PATH}")
print(f"\n✓ Notebook 02 completo — listo para 03_predictive_model")

In [0]:
# CELDA 5 — Reportes descriptivos para el dashboard (versión corregida)

import pandas as pd
import numpy as np

DESCRIPTIVE_PATH = f"{SILVER}/descriptive"
dbutils.fs.mkdirs(DESCRIPTIVE_PATH)

# Recargar dataset enriquecido
df = spark.read.parquet(PREDICT_FE_PATH).toPandas()

# Grados y secciones como strings (como están en los datos)
grados = sorted(df['grade'].unique().tolist())
secciones = sorted(df['section_anon'].unique().tolist())

print(f"✓ Dataset cargado: {len(df)} estudiantes")
print(f"  Grados: {grados}")
print(f"  Secciones: {secciones}")

# ── REPORTE 1 — Por asignatura ──
rows = []
for g in GROUPS:
    t1, t2, t3 = f'{g}_T1', f'{g}_T2', f'{g}_T3'
    for grado in grados:
        for seccion in secciones:
            mask = (df['grade'] == grado) & (df['section_anon'] == seccion)
            sub = df[mask]
            if len(sub) == 0:
                continue
            sub_mat = sub[sub[t1].notna()] if t1 in sub.columns else sub
            if len(sub_mat) == 0:
                continue

            row = {
                'materia': g,
                'grado': grado,
                'seccion': seccion,
                'n_estudiantes': len(sub_mat),
                'avg_T1': round(sub_mat[t1].mean(), 2) if t1 in sub_mat.columns else None,
                'avg_T2': round(sub_mat[t2].mean(), 2) if t2 in sub_mat.columns else None,
                'avg_T3': round(sub_mat[t3].mean(), 2) if t3 in sub_mat.columns else None,
                'n_bajo_T1': int((sub_mat[t1] < 4.0).sum()) if t1 in sub_mat.columns else 0,
                'n_bajo_T2': int((sub_mat[t2] < 4.0).sum()) if t2 in sub_mat.columns else 0,
                'n_bajo_T3': int((sub_mat[t3] < 4.0).sum()) if t3 in sub_mat.columns else 0,
                'pct_bajo_T2': round((sub_mat[t2] < 4.0).mean() * 100, 1) if t2 in sub_mat.columns else 0,
                'pct_bajo_T3': round((sub_mat[t3] < 4.0).mean() * 100, 1) if t3 in sub_mat.columns else 0,
                'delta_avg': round(sub_mat[t2].mean() - sub_mat[t1].mean(), 2) if t1 in sub_mat.columns and t2 in sub_mat.columns else None,
            }
            if all(c in sub_mat.columns for c in [t1, t2, t3]):
                acum = sub_mat[t1]*0.3 + sub_mat[t2]*0.3 + sub_mat[t3]*0.4
                row['avg_acumulada'] = round(acum.mean(), 2)
                row['n_bajo_acumulada'] = int((acum < 4.0).sum())
                row['pct_bajo_acumulada'] = round((acum < 4.0).mean() * 100, 1)
            rows.append(row)

df_por_asignatura = pd.DataFrame(rows)
print(f"\n✓ Reporte por asignatura: {len(df_por_asignatura)} filas")
print(df_por_asignatura[['materia','grado','seccion','n_estudiantes','avg_T2','pct_bajo_T2']].head(10).to_string())

spark.createDataFrame(df_por_asignatura).write.mode("overwrite").parquet(
    f"{DESCRIPTIVE_PATH}/por_asignatura"
)

# ── REPORTE 2 — Por estudiante individual ──
df_individual = df[['student_id', 'grade', 'section_anon']].copy()

for g in GROUPS:
    t1, t2, t3 = f'{g}_T1', f'{g}_T2', f'{g}_T3'
    df_individual[f'{g}_T1'] = df[t1] if t1 in df.columns else None
    df_individual[f'{g}_T2'] = df[t2] if t2 in df.columns else None
    df_individual[f'{g}_T3'] = df[t3] if t3 in df.columns else None
    if all(c in df.columns for c in [t1, t2, t3]):
        acum = df[t1]*0.3 + df[t2]*0.3 + df[t3]*0.4
        df_individual[f'{g}_acumulada'] = acum.round(2)
        df_individual[f'{g}_min_T3'] = df[f'{g}_min_T3'].round(2) if f'{g}_min_T3' in df.columns else None
        df_individual[f'{g}_bajo'] = (acum < 4.0)

# Variables para radar ABC
df_individual['pct_asistencia'] = np.where(
    df['total_absences'] > 0,
    (100 - (df['absence_class'] / df['total_absences'] * 100)).round(1),
    100
)
df_individual['indice_disciplinario'] = df['n_f1'] * 1 + df['n_f2'] * 3
df_individual['n_bajo_acumulada'] = df_individual[
    [f'{g}_bajo' for g in GROUPS if f'{g}_bajo' in df_individual.columns]
].sum(axis=1)
df_individual['tendencia_general'] = df['tendencia_general']
df_individual['n_f1'] = df['n_f1']
df_individual['n_f2'] = df['n_f2']
df_individual['total_absences'] = df['total_absences']
df_individual['absence_class'] = df['absence_class']
df_individual['late'] = df['late']
df_individual['early_leave'] = df['early_leave']
df_individual['risk_level'] = df['risk_level']
df_individual['n_subjects_below_4'] = df['n_subjects_below_4']
df_individual['avg_T1'] = df['avg_T1']
df_individual['avg_T2'] = df['avg_T2']
df_individual['dispersion_T2'] = df['dispersion_T2']

spark.createDataFrame(df_individual).write.mode("overwrite").parquet(
    f"{DESCRIPTIVE_PATH}/por_estudiante"
)
print(f"✓ Reporte por estudiante: {len(df_individual)} filas")

# ── REPORTE 3 — Resumen general ──
resumen_rows = []
for grado in ['todos'] + grados:
    for seccion in ['todas'] + secciones:
        if grado == 'todos':
            sub = df if seccion == 'todas' else df[df['section_anon'] == seccion]
        else:
            sub = df[df['grade'] == grado] if seccion == 'todas' else df[
                (df['grade'] == grado) & (df['section_anon'] == seccion)
            ]
        if len(sub) == 0:
            continue
        resumen_rows.append({
            'grado': str(grado),
            'seccion': seccion,
            'n_estudiantes': len(sub),
            'avg_asistencia': round(100 - sub['absence_class'].mean(), 1),
            'n_f1_promedio': round(sub['n_f1'].mean(), 2),
            'n_f2_promedio': round(sub['n_f2'].mean(), 2),
            'n_bajo_promedio': round(sub['n_subjects_below_4'].mean(), 2),
            'pct_critico': round((sub['risk_level'] == 'critical').mean() * 100, 1),
            'pct_recovery': round((sub['risk_level'] == 'recovery').mean() * 100, 1),
            'pct_no_risk': round((sub['risk_level'] == 'no_risk').mean() * 100, 1),
        })

df_resumen = pd.DataFrame(resumen_rows)
spark.createDataFrame(df_resumen).write.mode("overwrite").parquet(
    f"{DESCRIPTIVE_PATH}/resumen_general"
)
print(f"✓ Reporte resumen general: {len(df_resumen)} filas")

print("\n" + "="*55)
print("CELDA 5 COMPLETA")
print("="*55)
print(f"  silver/descriptive/por_asignatura  → Módulo Académico")
print(f"  silver/descriptive/por_estudiante  → Ficha + Radar ABC")
print(f"  silver/descriptive/resumen_general → Métricas globales")